# Data Science Bowl 2018 — U-Net (PyTorch) — Colab

Kaggle [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018) (세포핵 분할) 을 **PyTorch U-Net** 으로 푸는 기본 예제입니다. U-Net 구조와 세그멘테이션 학습 흐름을 공부하기에 좋습니다.

- 각 샘플 = 원본 이미지 1장 + 핵별 마스크 여러 장 → **하나의 이진 마스크**로 합쳐 학습합니다.
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 학습 후 검증 **Dice** 점수를 확인하고 예측 마스크를 시각화합니다.
- (선택) 마지막에 RLE 형식 제출 파일 `submission.csv` 를 `/content` 에 생성합니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU (CPU도 가능하나 느림).  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pillow", "matplotlib", "scikit-image"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 데이터 다운로드 (stage1_train / stage1_test)

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

for fn in ["stage1_train.zip", "stage1_test.zip"]:
    api.competition_download_file("data-science-bowl-2018", fn, path=WORK_DIR, quiet=False)

for fn, out in [("stage1_train.zip", "stage1_train"), ("stage1_test.zip", "stage1_test")]:
    zp = os.path.join(WORK_DIR, fn)
    with zipfile.ZipFile(zp) as zf: zf.extractall(os.path.join(WORK_DIR, out))
    os.remove(zp)

TRAIN_DIR = os.path.join(WORK_DIR, "stage1_train")
TEST_DIR = os.path.join(WORK_DIR, "stage1_test")
print("train samples:", len(os.listdir(TRAIN_DIR)), "| test samples:", len(os.listdir(TEST_DIR)))

## 2. 데이터 준비

이미지는 RGB 3채널, 마스크는 핵별 PNG 를 모두 합쳐 이진 마스크로 만든 뒤 `128×128` 로 리사이즈합니다.

In [ ]:
import numpy as np, glob
from PIL import Image

SZ = 128

def load_sample(sample_dir):
    img_path = glob.glob(os.path.join(sample_dir, "images", "*.png"))[0]
    img = np.asarray(Image.open(img_path).convert("RGB").resize((SZ, SZ)), dtype="float32") / 255.0
    mask = np.zeros((SZ, SZ), dtype="float32")
    for mp in glob.glob(os.path.join(sample_dir, "masks", "*.png")):
        m = np.asarray(Image.open(mp).convert("L").resize((SZ, SZ))) > 0
        mask = np.maximum(mask, m.astype("float32"))
    return img.transpose(2, 0, 1), mask[None, :, :]

ids = sorted(os.listdir(TRAIN_DIR))
X = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[0] for i in ids])
Y = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[1] for i in ids])
print("X:", X.shape, "| Y:", Y.shape, "| mask mean:", round(float(Y.mean()), 3))

# train / valid 분할
import torch
rng = np.random.RandomState(42); idx = rng.permutation(len(X)); cut = int(len(X) * 0.9)
tr_i, va_i = idx[:cut], idx[cut:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| train:", len(tr_i), "valid:", len(va_i))

## 3. U-Net 모델 정의

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.c(x)

class UNet(nn.Module):
    def __init__(self, ch=3, n_classes=1):
        super().__init__()
        self.d1 = DoubleConv(ch, 16); self.d2 = DoubleConv(16, 32); self.d3 = DoubleConv(32, 64)
        self.pool = nn.MaxPool2d(2); self.bottleneck = DoubleConv(64, 128)
        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2); self.c3 = DoubleConv(128, 64)
        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2); self.c2 = DoubleConv(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2); self.c1 = DoubleConv(32, 16)
        self.out = nn.Conv2d(16, n_classes, 1)
    def forward(self, x):
        c1 = self.d1(x); c2 = self.d2(self.pool(c1)); c3 = self.d3(self.pool(c2))
        b = self.bottleneck(self.pool(c3))
        x = self.c3(torch.cat([self.u3(b), c3], 1))
        x = self.c2(torch.cat([self.u2(x), c2], 1))
        x = self.c1(torch.cat([self.u1(x), c1], 1))
        return self.out(x)

model = UNet().to(device)
print("parameters:", sum(p.numel() for p in model.parameters()))

## 4. 학습 (BCE + Dice 손실)

In [ ]:
bce = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def dice_loss(logit, y):
    p = torch.sigmoid(logit)
    inter = (p * y).sum((1, 2, 3))
    return (1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)).mean()

def dice_score(logit, y):
    p = (torch.sigmoid(logit) > 0.5).float()
    inter = (p * y).sum()
    return ((2 * inter) / (p.sum() + y.sum() + 1e-6)).item()

Xt = torch.from_numpy(X); Yt = torch.from_numpy(Y)
EPOCHS = 25; BS = 16

for epoch in range(1, EPOCHS + 1):
    model.train(); perm = np.random.permutation(tr_i)
    for i in range(0, len(perm), BS):
        b = perm[i:i+BS]
        xb, yb = Xt[b].to(device), Yt[b].to(device)
        optimizer.zero_grad()
        out = model(xb); loss = bce(out, yb) + dice_loss(out, yb)
        loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        vd = np.mean([dice_score(model(Xt[va_i][j:j+BS].to(device)), Yt[va_i][j:j+BS].to(device))
                      for j in range(0, len(va_i), BS)])
    print(f"epoch {epoch:2d} | val dice = {vd:.3f}")

## 5. 예측 시각화

검증 샘플의 원본 / 정답 마스크 / 예측 마스크를 비교합니다.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
j = int(va_i[0])
with torch.no_grad():
    pred = torch.sigmoid(model(Xt[j:j+1].to(device)))[0, 0].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(X[j].transpose(1, 2, 0)); ax[0].set_title("Image")
ax[1].imshow(Y[j, 0], cmap="gray"); ax[1].set_title("Ground Truth")
ax[2].imshow(pred > 0.5, cmap="gray"); ax[2].set_title("Prediction")
for a in ax: a.axis("off")
plt.show()

## 6. (선택) 제출 파일 생성 — RLE

DSB2018 제출은 **핵 인스턴스별 Run-Length Encoding** 입니다. 예측 이진 마스크를 연결요소(connected components)로 나눠 각 핵을 RLE 로 인코딩합니다. (입문용 기본 버전)

In [ ]:
from skimage.morphology import label

def rle_encode(mask):
    pixels = mask.flatten(order="F")  # DSB2018: 열 우선(top->bottom)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

import pandas as pd, glob
rows = []
model.eval()
for sid in sorted(os.listdir(TEST_DIR)):
    img_path = glob.glob(os.path.join(TEST_DIR, sid, "images", "*.png"))[0]
    orig = Image.open(img_path).convert("RGB"); W, H = orig.size
    x = np.asarray(orig.resize((SZ, SZ)), dtype="float32").transpose(2, 0, 1) / 255.0
    with torch.no_grad():
        p = torch.sigmoid(model(torch.from_numpy(x[None]).to(device)))[0, 0].cpu().numpy()
    # 원본 크기로 복원
    pm = np.asarray(Image.fromarray((p > 0.5).astype("uint8") * 255).resize((W, H))) > 127
    lab = label(pm)
    found = False
    for k in range(1, lab.max() + 1):
        rle = rle_encode(lab == k)
        if rle:
            rows.append({"ImageId": sid, "EncodedPixels": rle}); found = True
    if not found:  # 핵 미검출 시 빈 예측 1행
        rows.append({"ImageId": sid, "EncodedPixels": ""})

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame(rows, columns=["ImageId", "EncodedPixels"]).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(rows))

## 7. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[Data Science Bowl 2018 제출 페이지](https://www.kaggle.com/c/data-science-bowl-2018/submit)**

- 대회 홈: https://www.kaggle.com/c/data-science-bowl-2018
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.